In [1]:
import pandas as pd
import numpy as np
import os
from collections import Counter
import re

# preprocessing mutation.tsv

In [2]:
fileName = r'../data/raw/mutations.tsv'
df = pd.read_csv(fileName, sep='\t', keep_default_na=False)
print(df.shape)
df.head()

(58250, 15)


,#Feature AC,Feature short label,Feature range(s),Original sequence,Resulting sequence,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC
0,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],83-83,L,A,mutation(MI:0118),,uniprotkb:P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418
1,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],87-87,L,A,mutation(MI:0118),,uniprotkb:P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418
2,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],91-91,L,A,mutation(MI:0118),,uniprotkb:P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418
3,EBI-6925687,p.Cys169Ser,169-169,C,S,mutation(MI:0118),,uniprotkb:P0A6H1,clpX,,83333 - Escherichia coli (strain K12),"uniprotkb:P0A6H1(protein(MI:0326), 83333 - Esc...",23622246,Supp Fig. 2A,EBI-6925660
4,EBI-6898360,p.Phe508del,508-508,F,.,mutation(MI:0118),,uniprotkb:P13569,CFTR,,9606 - Homo sapiens,"uniprotkb:P13569(protein(MI:0326), 9606 - Homo...",22038833,"1B, 4B",EBI-6898336


In [3]:
df[(df['Feature annotation'].str.contains('high-throughput'))].shape

(11028, 15)

In [4]:
# drop high-throught
df = df[~(df['Feature annotation'].str.contains('high-throughput'))]
df.shape

(47222, 15)

## drop entries with >2 participants, and drop entries that the number of partner don't match the number of uniprotAC. (to filter binary protein-protein interaction)

In [5]:
import re
p = re.compile(r'uniprotkb:(.*?)[(]', re.S)
partner = []
n_partner = []
count = 0
for i in df['Interaction participants']:
    tmp = re.findall(p, i)
#     num = re.findall(p2, i)
    num = i.count(';') + 1
    partner.append(tmp)
    n_partner.append(num)
df['partners'] = partner
df['n_partner'] = n_partner

df = df[df['n_partner'] < 3]
print('after delete items with more than 2 partners {}'.format(df.shape))
df = df[df['partners'].apply(lambda x: len(x)) == df['n_partner']]
print('after delete items with not identical number of partners and n_partner {}'.format(df.shape))

after delete items with more than 2 partners (43126, 17)
after delete items with not identical number of partners and n_partner (35267, 17)


## drop entries with same interactionAC but different affected protein AC (drop same interaction with multiple mutations)

In [6]:
df1 = df[df.duplicated(['Affected protein AC', 'Interaction AC'], keep=False)] # choose items with same interactAC-aff pro AC pair
df2 = df.drop_duplicates(['Interaction AC'], keep=False) # choose items with only one time interactionAC
df = pd.concat([df1, df2])
print(df.shape)

(34712, 17)


## drop entries without uniprotAC

In [7]:
df = df[df['Affected protein AC'].str.contains('uniprotkb:', na=False)]
print(df.shape)

(34696, 17)


## simplify uniprotkb label

In [8]:
df['Affected protein AC'] = df['Affected protein AC'].str.replace('uniprotkb:', '')
df.head()

,#Feature AC,Feature short label,Feature range(s),Original sequence,Resulting sequence,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC,partners,n_partner
0,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],83-83,L,A,mutation(MI:0118),,P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418,"[P03243-1, F1M589]",2
1,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],87-87,L,A,mutation(MI:0118),,P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418,"[P03243-1, F1M589]",2
2,EBI-6915452,p.[Leu83Ala;Leu87Ala;Leu91Ala],91-91,L,A,mutation(MI:0118),,P03243-1,p03243-1,,28285 - Human adenovirus C serotype 5 (HAdV-5),"uniprotkb:P03243-1(protein(MI:0326), 28285 - H...",20639899,Fig. 1d,EBI-2941418,"[P03243-1, F1M589]",2
5,EBI-6925862,p.Cys169Ser,169-169,C,S,mutation(MI:0118),,P0A6H1,clpX,,83333 - Escherichia coli (strain K12),"uniprotkb:P0A6H1(protein(MI:0326), 83333 - Esc...",23622246,Supp Fig. 2C,EBI-6925855,"[P0A6H1, P0A6H1]",2
13,EBI-8875481,p.Ile204Tyr,204-204,I,Y,mutation(MI:0118),,Q8TE30,q8te30_human,,9606 - Homo sapiens,"uniprotkb:Q8TE30(protein(MI:0326), 9606 - Homo...",24267889,"Fig. 6F, Supp. Fig. 7H",EBI-8875425,"[Q8TE30, Q9UN81]",2


## delete 'mutation' feature type

In [9]:
df = df[~df['Feature type'].isin(['mutation(MI:0118)'])]
print(df.shape)

(31792, 17)


## delete non- regular acid items with same featureAC

In [10]:
f_ = df[df['Resulting sequence'].str.contains('B|J|O|Z', na=False)]['#Feature AC'].tolist()
df = df[~df['#Feature AC'].isin(f_)]
print(df.shape)
print(f_)

(31791, 17)
['EBI-8291032']


## delete 'PRO_' uniprotAC in table

In [11]:
df = df[~df['Affected protein AC'].str.contains('PRO_')]
df.shape

(31488, 17)

## get all sequence from uniprot (prepare for uniprot fasta retrieve https://www.uniprot.org/uploadlists/)

In [12]:
def flatlist(acList):
    return [item for sublist in acList for item in sublist]

ac1 = set(flatlist(df['partners'].values.tolist()))
ac2 = set(df['Affected protein AC'].values.tolist())
acAll = ac1 | ac2
with open('../data/middlefile/acAll.txt', 'w') as f:
    for x in acAll:
        f.write(x + '\n')


In [13]:
p = re.compile('PRO_')
acAll = [x for x in acAll if not p.findall(x)]
len(acAll)

7574

In [14]:
ac = []
info = []
seq = []
seqline = ''
initFlag = True
fastaFile = '../data/raw/allAC.fasta' # from uniprot website mapping, download with canonical and isoform
with open(fastaFile, 'r') as f:
    for line in f:
        line = line.strip()
        if '>' in line:
            res = re.findall(r'\|([^"]+)\|', line)[0]
            ac.append(res)
            info.append(line)
            if initFlag:
                initFlag = False
            else:
                seq.append(seqline)
                seqline = ''
        else:
            seqline += line
    seq.append(seqline)
fastaTable = pd.DataFrame({'ac': ac, 'info': info, 'seq': seq})

In [15]:
import pandas as pd

dup_counts = fastaTable['ac'].value_counts()
duplicates_ac = dup_counts[dup_counts > 1].index.tolist()

print(len(duplicates_ac))

for ac in duplicates_ac:
    seqs = fastaTable.loc[fastaTable['ac'] == ac, 'seq']
    if seqs.nunique() > 1:
        print(ac)
fastaTable = fastaTable.drop_duplicates(subset=['ac'], keep='first')

351


In [39]:
outFasta = '../data/middle_file/allAC_dedup.fasta'
with open(outFasta, 'w') as f:
    for idx, row in fastaTable.iterrows():
        f.write(f"{row['info']}\n")
        sequence = row['seq']
        for i in range(0, len(sequence), 60):
            f.write(sequence[i:i+60] + '\n')

'''
# Run the following command to obtain the clustering results.
conda install -c conda-forge -c bioconda mmseqs2 -y

mkdir -p ../data/middlefile/mmseqs_cluster/tmp
mkdir -p ../data/middlefile/mmseqs_cluster/results

mmseqs easy-cluster \
    ../data/middlefile/allAC_dedup.fasta \
    ../data/middlefile/mmseqs_cluster/results/allAC_dedup_cluster \
    ../data/middlefile/mmseqs_cluster/tmp \
    --min-seq-id 0.4 \
    -c 0.8 \
    --cov-mode 1 \
    --threads 16
'''

## select valid uniprotAC to make following selection

In [16]:
validAC1 = fastaTable[fastaTable['ac'].isin(acAll)]

validAC2 = fastaTable[~fastaTable['ac'].isin(acAll)]
acAll_series = pd.Series(list(acAll))
validAC2 = validAC2[validAC2['ac'].isin(acAll_series.str.split('-', expand=True)[0])]
validAC = pd.concat([validAC1, validAC2])
print(validAC1.shape)
print(validAC2.shape)
print(validAC.shape)

(6847, 3)
(327, 3)
(7174, 3)


## make the 'affected protein AC' - 'uniprotAC' dict. Some have 'multiple key' -> 'single value' relationship eg: apac['P19838-1'] = 'P19838', apac['P19838'] = 'P19838'

In [17]:
apacKey = []
acValue = []
for ac in acAll:
    if ac in validAC['ac'].values:
        apacKey.append(ac)
        acValue.append(ac)
    elif ac.split('-')[0] in validAC['ac'].values:
        apacKey.append(ac)
        acValue.append(ac.split('-')[0])
apac2ac = dict(zip(apacKey, acValue))


### transform all isoform AC in table into real uniprotAC(canonical with no isoform '-'), eg: O43889-2 ->O43889, O43889-3 -> O43889-3

In [18]:
df = df[df['Affected protein AC'].isin(apac2ac.keys())]

In [19]:
df = df[df['partners'].apply(lambda x: set(x) < set(list(apac2ac.keys())))]

## make interaction with multi position mutations into one 

In [20]:
# pos = df['Feature range(s)'].str.split('-', expand=True)
# df['start'] = pos[0]
# df['end'] = pos[1]

comCol = df.columns.tolist()
comCol.remove('Feature range(s)')
comCol.remove('Original sequence')
comCol.remove('Resulting sequence')

df_1 = df.groupby('#Feature AC')[['Feature range(s)','Original sequence', 'Resulting sequence']].agg(list)
df_2 = df[comCol].drop_duplicates('#Feature AC', keep='first')
df = pd.merge(df_1, df_2, on = '#Feature AC')
df.reset_index(drop=True, inplace=True)
df.shape

(26816, 17)

In [21]:
df.head()

,#Feature AC,Feature range(s),Original sequence,Resulting sequence,Feature short label,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,Affected protein organism,Interaction participants,PubMedID,Figure legend,Interaction AC,partners,n_partner
0,EBI-10039489,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039307,"[Q03694, P28795]",2
1,EBI-10039495,[188-188],[N],[I],p.Asn188Ile,mutation decreasing(MI:0119),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:Q03694(protein(MI:0326), 559292 - Sa...",23900285,f1c,EBI-10039491,"[Q03694, P28795]",2
2,EBI-10039551,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2a f2b,EBI-10039532,"[P28795, Q03694]",2
3,EBI-10039706,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2c,EBI-10039697,"[P28795, Q03694]",2
4,EBI-10039722,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,559292 - Saccharomyces cerevisiae,"uniprotkb:P28795(protein(MI:0326), 559292 - Sa...",23900285,f2d,EBI-10039716,"[P28795, Q03694]",2


## make mutprotein seq and participants seq

In [22]:
validAC_index = validAC.copy()
validAC_index = validAC_index.set_index('ac')

In [23]:
mutAC = [apac2ac[x] for x in df['Affected protein AC']]
mut0 = []
for i in mutAC:
    mut0.append(validAC_index.loc[i, 'seq'])

In [24]:
par = []
parAC = []

for i in df.index:
    sameFlag = False
    if len(df.loc[i, 'partners']) > 1:
        for j in df.loc[i, 'partners']:
            if j != df.loc[i, 'Affected protein AC']:
                par.append(validAC_index.loc[apac2ac[j], 'seq'])
                parAC.append(apac2ac[j])
            elif sameFlag:
                par.append(validAC_index.loc[apac2ac[j], 'seq'])
                parAC.append(apac2ac[j])
            else:
                sameFlag = True
    elif len(df.loc[i, 'partners']) == 1:
        par.append(validAC_index.loc[apac2ac[df.loc[i, 'partners'][0]], 'seq'])
        parAC.append(apac2ac[df.loc[i, 'partners'][0]])
print(len(par))
print(df.shape)

26816
(26816, 17)


In [25]:
df['mutAC'] = mutAC
df['mut0'] = mut0
df['parAC'] = parAC
df['par0'] = par

In [26]:
mut1 = []
for i in df.index:
    tmp = df.loc[i, 'mut0']
    for j in range(len(df.loc[i, 'Feature range(s)'])):
        pos0 = int(df.loc[i, 'Feature range(s)'][j].split('-')[0])
        pos1 = int(df.loc[i, 'Feature range(s)'][j].split('-')[1])
        ori = df.loc[i, 'Original sequence'][j]
        mut = df.loc[i, 'Resulting sequence'][j]
        if tmp[(pos0 - 1): pos1] != ori:
            print(df.loc[i, 'Affected protein AC'])
#             mut1.append('error_match')
            continue
        else:
            tmp = tmp[:(pos0 - 1)] + mut + tmp[pos1:]
    tmp = tmp.replace('.', '')
    mut1.append(tmp)
print(len(mut1))

O95793-2
P09651-2
O08573-2
O08573-2
Q99697-3
Q99697-3
Q99697-3
Q13627-2
Q92851-4
Q7Z2E3-7
Q7Z2E3-7
Q7Z2E3-7
P05549-5
P05549-5
P05549-5
Q15583-2
Q15583-2
Q15583-2
Q15583-2
Q15583-2
P43897-2
P08949-2
Q8WWX8-3
P43897-2
P43897-2
O14836-2
O14836-2
O14836-2
O14836-2
O14836-2
P29372-4
P60201-2
P60201-2
P07510-2
P04075-2
P04075-2
P04075-2
P12104
P07510-2
P07510-2
P51814-6
P51814-6
P51814-6
P07510-2
P07510-2
P22557-2
P22557-2
P22557-2
P51795
P60201-2
P60201-2
P60201-2
P60201-2
P07510-2
P07510-2
Q15583-2
Q15583-2
Q15583-2
Q15583-2
Q15583-2
P07510-2
Q92851-4
Q7Z2E3-7
Q7Z2E3-7
Q7Z2E3-7
P05549-5
P05549-5
P05549-5
Q15583-2
Q15583-2
Q15583-2
Q15583-2
Q15583-2
P43897-2
P08949-2
Q8WWX8-3
P43897-2
P43897-2
O14836-2
O14836-2
O14836-2
O14836-2
O14836-2
P29372-4
P60201-2
P60201-2
P07510-2
P04075-2
P04075-2
P04075-2
P12104
P07510-2
P07510-2
P51814-6
P51814-6
P51814-6
P07510-2
P07510-2
P22557-2
P22557-2
P22557-2
P51795
P60201-2
P60201-2
P60201-2
P60201-2
P07510-2
P07510-2
Q15583-2
Q15583-2
Q15583-2
Q15583-2


In [27]:
df['mut1'] = mut1

In [28]:
df['label'] = 2
df.loc[df['Feature type'].str.contains('disrupting'), 'label'] = 0
df.loc[df['Feature type'].str.contains('decreasing'), 'label'] = 1
df.loc[df['Feature type'].str.contains('increasing'), 'label'] = 3
df.loc[df['Feature type'].str.contains('causing'), 'label'] = 4

In [29]:
df['mutAC1'] = df['mutAC'] + '_' + df['Feature short label']
df['mutAC1'] = df['mutAC1'].str.replace('_p.', '_')
df['mutAC1'] = df['mutAC1'].str.replace('[', '-')
df['mutAC1'] = df['mutAC1'].str.replace(']', '-')
df['mutAC1'] = df['mutAC1'].str.replace(';', '_')

In [30]:
df['Feature type'].value_counts()

Feature type
mutation with no effect(MI:2226)         8111
mutation disrupting strength(MI:1128)    5073
mutation disrupting(MI:0573)             4388
mutation decreasing(MI:0119)             4189
mutation decreasing strength(MI:1133)    2341
mutation increasing(MI:0382)             1112
mutation increasing strength(MI:1132)     613
mutation disrupting rate(MI:1129)         350
mutation decreasing rate(MI:1130)         275
mutation causing(MI:2227)                 222
mutation increasing rate(MI:1131)         142
Name: count, dtype: int64

In [31]:
df['label'].value_counts()

label
0    9811
2    8111
1    6805
3    1867
4     222
Name: count, dtype: int64

In [32]:
df.head()

,#Feature AC,Feature range(s),Original sequence,Resulting sequence,Feature short label,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,...,Interaction AC,partners,n_partner,mutAC,mut0,parAC,par0,mut1,label,mutAC1
0,EBI-10039489,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,EBI-10039307,"[Q03694, P28795]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,0,P28795_Val81Glu
1,EBI-10039495,[188-188],[N],[I],p.Asn188Ile,mutation decreasing(MI:0119),,P28795,PEX3,,...,EBI-10039491,"[Q03694, P28795]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,1,P28795_Asn188Ile
2,EBI-10039551,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,EBI-10039532,"[P28795, Q03694]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,0,P28795_Val81Glu
3,EBI-10039706,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,EBI-10039697,"[P28795, Q03694]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,0,P28795_Val81Glu
4,EBI-10039722,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,EBI-10039716,"[P28795, Q03694]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,0,P28795_Val81Glu


## delete items with same participants (for unknown mutated or wildtype in the interaction)

In [33]:
df = df[~(df['mutAC'] == df['parAC'])]
print('after drop same participants items: {}'.format(df.shape))

after drop same participants items: (23802, 24)


## drop unregular aa

In [34]:
df = df[~(df['mut0'].str.contains('B|J|O|U|X|Z'))]
df = df[~(df['mut1'].str.contains('B|J|O|U|X|Z'))]
df = df[~(df['par0'].str.contains('B|J|O|U|X|Z'))]
print('after drop unregular aa: {}'.format(df.shape))

after drop unregular aa: (23762, 24)


In [38]:
df.to_pickle('../data/processed/processed_mutations.dataset')

In [37]:
df.head()

,#Feature AC,Feature range(s),Original sequence,Resulting sequence,Feature short label,Feature type,Feature annotation,Affected protein AC,Affected protein symbol,Affected protein full name,...,Interaction AC,partners,n_partner,mutAC,mut0,parAC,par0,mut1,label,mutAC1
0,EBI-10039489,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,EBI-10039307,"[Q03694, P28795]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,0,P28795_Val81Glu
1,EBI-10039495,[188-188],[N],[I],p.Asn188Ile,mutation decreasing(MI:0119),,P28795,PEX3,,...,EBI-10039491,"[Q03694, P28795]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,1,P28795_Asn188Ile
2,EBI-10039551,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,EBI-10039532,"[P28795, Q03694]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,0,P28795_Val81Glu
3,EBI-10039706,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,EBI-10039697,"[P28795, Q03694]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,0,P28795_Val81Glu
4,EBI-10039722,[81-81],[V],[E],p.Val81Glu,mutation disrupting(MI:0573),,P28795,PEX3,,...,EBI-10039716,"[P28795, Q03694]",2,P28795,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,Q03694,MVLSRGETKKNSVRLTAKQEKKPQSTFQTLKQSLKLSNNKKLKQDS...,MAPNQRSRSLLQRHRGKVLISLTGIAALFTTGSVVVFFVKRWLYKQ...,0,P28795_Val81Glu
